In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Preparing compressed dataset

In [ ]:
folder = "/content/drive/MyDrive/PCA_Models"
BB = "/BB"
PB = "/PB"
VB = "/VB"

In [ ]:
datapath = "/content/drive/MyDrive/resampled_last/15sec/EBX"

# Pick data and process and Save

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import tqdm

In [ ]:
def join_with_top_padding(df_left, df_right):
    max_len = max(len(df_left), len(df_right))

    def pad(df):
        pad_len = max_len - len(df)
        if pad_len > 0:
            top = pd.DataFrame([[np.nan]*df.shape[1]] * pad_len, columns=df.columns)
            df = pd.concat([top, df], ignore_index=True)
        return df

    df_left  = pad(df_left)
    df_right = pad(df_right)

    return pd.concat([df_left, df_right], axis=1)


In [ ]:
!mkdir "/content/15_sec_last_PCAcompressed"

mkdir: cannot create directory ‘/content/15_sec_last_PCAcompressed’: File exists


In [ ]:
dffff = pd.DataFrame([1, 7, 20])
dffff.diff()

,0
0,NaN
1,6.0
2,13.0


In [ ]:
for i in tqdm.tqdm(range(510), desc="Processing"):

  if(i >= 60 and i <= 80):
    continue

  df = pd.read_csv(datapath+"/day"+str(i)+".csv")
  out_df = df['close'].diff()
  out_df = pd.concat([out_df, pd.DataFrame([0])], axis = 0, ignore_index = True)
  for j in ["PB", "BB", "VB"]:
    for k in range(2, 13):
      col_select = []
      if(j == "PB"):
        col_select = ["PB"+str(i)+"_T"+str(k) for i in range(1, 19)]
      elif(j == "BB"):
        col_select = ["BB"+str(i)+"_T"+str(k) for i in range(4, 16)] + ["BB24", "BB25"]
      elif(j == "VB"):
        col_select = ["VB"+str(i)+"_T"+str(k) for i in range(1, 7)]
      df_temp = df.loc[:, col_select]
      loaded_model = load_model(folder + "/" + j + "/" + j + "_T" + str(k) + "_" + (str(4) if j=="PB" else str(3)) + "col.keras")
      df_temp = df_temp.dropna()
      X = df_temp.values
      X = (X - np.mean(X, axis=0))/np.std(X, axis=0)
      out = loaded_model(X)
      df_temp = pd.DataFrame(out)
      out_df = join_with_top_padding(out_df, df_temp)
  out_df.to_csv("/content/15_sec_last_PCAcompressed/day"+str(i)+".csv")


Processing: 100%|██████████| 510/510 [26:21<00:00,  3.10s/it]


In [ ]:
!mv "/content/15_sec_last_PCAcompressed" "/content/drive/MyDrive/"